In [ ]:
import os
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [2]:
from pypdf import PdfReader

def load_pdf(file_path):
    """
    Reads the text content from a PDF file and returns it as a single string.

    Parameters:
    - file_path (str): The file path to the PDF file.

    Returns:
    - str: The concatenated text content of all pages in the PDF.
    """
    reader = PdfReader(file_path)

    text = ""
    for page in reader.pages:
        text += page.extract_text()

    return text

pdf_text = load_pdf(file_path="../assets/data/BRTS_Chatbot_Dataset.pdf")

In [3]:
import re

def split_text(text: str):
    """
    Splits a text string into a list of non-empty substrings based on the specified pattern.
    The "\n \n" pattern will split the document para by para
    Parameters:
    - text (str): The input text to be split.

    Returns:
    - List[str]: A list containing non-empty substrings obtained by splitting the input text.

    """
    split_text = re.split('\n \n', text)
    return [i for i in split_text if i != ""]

chunked_text = split_text(text=pdf_text)

In [4]:
import google.generativeai as genai
from chromadb import Documents, EmbeddingFunction, Embeddings
import os

class GeminiEmbeddingFunction(EmbeddingFunction):
    """
    Custom embedding function using the Gemini AI API for document retrieval.

    This class extends the EmbeddingFunction class and implements the __call__ method
    to generate embeddings for a given set of documents using the Gemini AI API.

    Parameters:
    - input (Documents): A collection of documents to be embedded.

    Returns:
    - Embeddings: Embeddings generated for the input documents.
    """
    def __call__(self, input: Documents) -> Embeddings:
        gemini_api_key = os.getenv("GEMINI_API_KEY")
        if not gemini_api_key:
            raise ValueError("Gemini API Key not provided. Please provide GEMINI_API_KEY as an environment variable")
        genai.configure(api_key=gemini_api_key)
        model = "models/embedding-001"
        title = "Custom query"
        return genai.embed_content(model=model,
                                   content=input,
                                   task_type="retrieval_document",
                                   title=title)["embedding"]

c:\Users\arpy8\repos\Bhopal-BRTS-ChatBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import chromadb
from typing import List

def create_chroma_db(documents:List, path:str, name:str):
    """
    Creates a Chroma database using the provided documents, path, and collection name.

    Parameters:
    - documents: An iterable of documents to be added to the Chroma database.
    - path (str): The path where the Chroma database will be stored.
    - name (str): The name of the collection within the Chroma database.

    Returns:
    - Tuple[chromadb.Collection, str]: A tuple containing the created Chroma Collection and its name.
    """
    chroma_client = chromadb.PersistentClient(path=path)
    db = chroma_client.create_collection(name=name, embedding_function=GeminiEmbeddingFunction())

    for i, d in enumerate(documents):
        db.add(documents=d, ids=str(i))

    return db, name

db,name =create_chroma_db(documents=chunked_text, 
                          path=r"C:\Users\arpy8\repos\Bhopal-BRTS-ChatBot\misc",
                          name="rag_experiment3")

In [6]:
def load_chroma_collection(path, name):
    """
    Loads an existing Chroma collection from the specified path with the given name.

    Parameters:
    - path (str): The path where the Chroma database is stored.
    - name (str): The name of the collection within the Chroma database.

    Returns:
    - chromadb.Collection: The loaded Chroma Collection.
    """
    chroma_client = chromadb.PersistentClient(path=path)
    db = chroma_client.get_collection(name=name, embedding_function=GeminiEmbeddingFunction())

    return db

db=load_chroma_collection(path=r"C:\Users\arpy8\repos\Bhopal-BRTS-ChatBot\misc", name="rag_experiment3")

In [7]:
def get_relevant_passage(query, db, n_results):
  passage = db.query(query_texts=[query], n_results=n_results)['documents'][0]
  return passage

relevant_text = get_relevant_passage(query="brts",db=db,n_results=3)
relevant_text

Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


["BRTS Chatbot Dat aset\n1\nBRTS Chatbot Dataset\nWhat is BRTS System\nBhopal BRTS was a bus rapid transit system located in Bhopal, Madhya \nPradesh, India, funded by the Central Government under its flagship JnNURM \nprogram. Unlike most BRT projects in India designed to serve suburban areas, \nBhopal BRTS primarily catered to the needs of the Central Business Districts \n\ue081CBDs). It launched in 2006 with a fleet of just 30 buses, following the JnNURM \nsanction, and expanded to 225 buses, including both AC and non-AC low-floor \nmodels. However, on December 26, 2023, the Government of Madhya Pradesh, \nled by Chief Minister Mohan Yadav, decided to discontinue the BRTS project \ndue to traffic issues caused by the corridor. Instead, a central road divider was \nplanned to separate traffic between the two lanes. The process of dismantling \nthe corridor began on January 20, 2024.\nHow does the BRTS system work?\nBhopal's Bus Rapid Transit System \ue081BRTS\ue082 is designed to enh

In [8]:
def make_rag_prompt(query, relevant_passage):
  escaped = relevant_passage.replace("'", "").replace('"', "").replace("\n", " ")
  prompt = ("""You are a helpful and informative bot that answers questions using text from the reference passage included below. \
  Be sure to respond in a complete sentence, being comprehensive, including all relevant background information. \
  However, you are talking to a non-technical audience, so be sure to break down complicated concepts and \
  strike a friendly and converstional tone. \
  If the passage is irrelevant to the answer, you may ignore it.
  QUESTION: '{query}'
  PASSAGE: '{relevant_passage}'

  ANSWER:
  """).format(query=query, relevant_passage=escaped)

  return prompt

In [9]:
import google.generativeai as genai
def __generate_answer(prompt):
    gemini_api_key = os.getenv("GEMINI_API_KEY")
    if not gemini_api_key:
        raise ValueError("Gemini API Key not provided. Please provide GEMINI_API_KEY as an environment variable")
    genai.configure(api_key=gemini_api_key)
    model = genai.GenerativeModel('gemini-1.5-flash')
    answer = model.generate_content(prompt)
    return answer.text

In [10]:
def generate_answer(db,query):
    #retrieve top 3 relevant text chunks
    relevant_text = get_relevant_passage(query,db,n_results=3)
    prompt = make_rag_prompt(query, 
                             relevant_passage="".join(relevant_text)) # joining the relevant chunks to create a single passage
    answer = __generate_answer(prompt)

    return answer

In [ ]:
db=load_chroma_collection(path=r"C:\Users\arpy8\repos\Bhopal-BRTS-ChatBot\misc",name="rag_experiment3")

answer = generate_answer(db, query="ticket price")
print(answer)

Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


The fare structure for the Bhopal Bus Rapid Transit System (BRTS) is tiered, based on the distance traveled.  For a short trip, the minimum fare starts around ₹10, and longer journeys can cost up to ₹30 or more, depending on the route and type of bus (AC or non-AC). 



In [ ]:
import google.generativeai as genai

def __generate_answer(prompt):
    gemini_api_key = os.getenv("GEMINI_API_KEY")
    if not gemini_api_key:
        raise ValueError("Gemini API Key not provided. Please provide GEMINI_API_KEY as an environment variable")
    genai.configure(api_key=gemini_api_key)
    model = genai.GenerativeModel('gemini-pro')
    answer = model.generate_content(prompt)
    return answer.text

__generate_answer("who are you")

'I am Gemini, a multimodal AI model, developed by Google. I am designed to provide information and assist users to the best of my abilities.'